# example / onnx / linear

**Intent**
- Takes `onnx_in`, a number.
- Outputs `onnx_out`, the input scaled by a fitted gain plus a fitted offset.
- Shows a fitted model end to end, with every number reproducible.
- Acceptance holds the rmse on `validation` and `backtest` to 0.5.
- It is a fixture, tested by `fab unittest` and `fab systest` and never released.

**Data**
- `training` is ten labelled points on a line with fixed noise, the only data the fit sees.
- `validation` is five more points, and `backtest` is ten further out, held back for acceptance.
- `golden` is six exactly representable inputs, then a request with the input missing that must give a signature error.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from os.path import join

import numpy as np

from workbench import setup_workspace

ws = setup_workspace("example/onnx/linear", {"kind": "embedded", "language": "rust"})


INPUT = ws.inputs[0]
OUTPUT = ws.outputs[0]

## Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-data))

In [ ]:
from workbench import data_load

NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [float(x) for x in range(first, first + count)]
    return {INPUT: xs, OUTPUT: [round(2.5 * x + 1.0 + NOISE[int(x) % 10], 4) for x in xs]}


training_pl, training_path = data_load(ws, "training", "onnx_linear_training", run=1, source="builder", build=lambda: labelled(0, 10))

validation_pl, validation_path = data_load(ws, "validation", "onnx_linear_validation", run=1, source="builder", build=lambda: labelled(10, 5))

backtest_pl, backtest_path = data_load(ws, "backtest", "onnx_linear_backtest", run=1, source="builder", build=lambda: labelled(15, 10))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Train ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-train))

In [ ]:
gain, offset = np.polyfit(training_pl[INPUT].to_numpy(), training_pl[OUTPUT].to_numpy(), 1)
GAIN, OFFSET = round(float(gain), 1), round(float(offset), 1)
print("fitted gain", GAIN, "offset", OFFSET)
assert (GAIN, OFFSET) == (2.5, 1.0), "the fit did not recover the line that made the data"

## Evaluate ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-evaluate))

In [ ]:
import onnx
from onnx import TensorProto, helper, numpy_helper

from workbench import evaluate_check, evaluate_measure, evaluate_predict

ONNX_OPSET = 21
ONNX_IR_VERSION = 10

graph = helper.make_graph(
    [helper.make_node("Mul", [INPUT, "gain"], ["scaled"]), helper.make_node("Add", ["scaled", "offset"], [OUTPUT])],
    "linear",
    [helper.make_tensor_value_info(INPUT, TensorProto.DOUBLE, [1, 1])],
    [helper.make_tensor_value_info(OUTPUT, TensorProto.DOUBLE, [1, 1])],
    initializer=[numpy_helper.from_array(np.array(GAIN, dtype=np.float64), "gain"), numpy_helper.from_array(np.array(OFFSET, dtype=np.float64), "offset")],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", ONNX_OPSET)])
model.ir_version = ONNX_IR_VERSION
onnx.checker.check_model(model)

model_path = join(ws.path, "model.onnx")
onnx.save(model, model_path)

predict = evaluate_predict(model_path, "onnx", ws.signature)


def acceptance_checks():
    checks = []
    for split, frame in (("validation", validation_pl), ("backtest", backtest_pl)):
        predicted = [predict({INPUT: [x]})[OUTPUT][0] for x in frame[INPUT].to_list()]
        checks.append({"split": split, "output": OUTPUT, "name": "rmse", "value": evaluate_measure(frame[OUTPUT].to_list(), predicted)["rmse"], "limit": 0.5, "direction": "max"})
    return checks


evaluation = evaluate_check(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "limit", entry["limit"], "passed", entry["passed"])

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench import package_workspace


def golden_samples():
    xs = [-1.0, 0.0, 0.25, 1.0, 2.5, 10.0]
    return {INPUT: [*xs, None], OUTPUT: [*[GAIN * x + OFFSET for x in xs], None], "expect": ["ok"] * len(xs) + ["error"]}


golden = golden_samples()
package_workspace(
    ws,
    "onnx",
    golden,
    trained_on="synthetic line, training split",
    inputs=[training_path],
    notes="least squares line, gain and offset built into the graph",
    implementation="onnx.helper graph, opset 21, ir version 10",
    model_path=model_path,
    evaluation=evaluation,
    tolerance={OUTPUT: {"absolute": 1e-9}},
    sequence="across zero, fractions and a larger value",
)